# OSS4008 Phase III — Residential AI Mapper (updated launcher)

**Run these cells from top to bottom in Google Colab.** This version uses a Cloudflare temporary tunnel because the Colab preview produced a Streamlit WebSocket loading loop. The ZIP is saved to Google Drive so future runtime resets do not require repeated uploads. The temporary tunnel URL is public; do not enter passwords or private information into the app.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/OSS4008_Phase_III')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_IN_DRIVE = PROJECT_DIR / 'OSS4008_Phase_III_Updated.zip'
print('Permanent project folder:', PROJECT_DIR)


In [ ]:
# On first use, upload the UPDATED ZIP file (not four separate files).
# On later sessions, this cell skips uploading because the ZIP is already in Drive.
if not ZIP_IN_DRIVE.exists():
    from google.colab import files
    print('Select OSS4008_Phase_III_Updated.zip from Downloads')
    uploaded = files.upload()
    if 'OSS4008_Phase_III_Updated.zip' not in uploaded:
        raise FileNotFoundError('Please upload OSS4008_Phase_III_Updated.zip')
    ZIP_IN_DRIVE.write_bytes(uploaded['OSS4008_Phase_III_Updated.zip'])
print('Saved ZIP:', ZIP_IN_DRIVE, 'size:', ZIP_IN_DRIVE.stat().st_size)


In [ ]:
import zipfile
APP_DIR = Path('/content/oss4008_phase3')
APP_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_IN_DRIVE) as z:
    for name in ['app.py', 'requirements.txt', 'README.md']:
        (APP_DIR / name).write_bytes(z.read('OSS4008_Phase_III_Updated/' + name))
assert (APP_DIR/'app.py').exists() and (APP_DIR/'requirements.txt').exists()
print('App ready:', APP_DIR)


In [ ]:
%pip -q install -r {APP_DIR / 'requirements.txt'}
print('Dependencies installed')


In [ ]:
import subprocess, time, requests
from pathlib import Path
# Avoid multiple servers when re-running a cell in the same runtime.
if 'streamlit_proc' in globals() and streamlit_proc.poll() is None:
    streamlit_proc.terminate()
    streamlit_proc.wait(timeout=10)
streamlit_log = open('/content/oss4008_streamlit.log', 'w')
streamlit_proc = subprocess.Popen([
    'streamlit', 'run', str(APP_DIR/'app.py'),
    '--server.address=127.0.0.1', '--server.port=8501',
    '--server.headless=true', '--server.enableCORS=false',
    '--server.enableXsrfProtection=false',
], stdout=streamlit_log, stderr=subprocess.STDOUT)
health = None
for _ in range(30):
    if streamlit_proc.poll() is not None:
        break
    try:
        r = requests.get('http://127.0.0.1:8501/_stcore/health', timeout=2)
        if r.status_code == 200:
            health = r.text
            break
    except requests.RequestException:
        pass
    time.sleep(1)
print('Streamlit running:', streamlit_proc.poll() is None, 'Health:', health)
if health != 'ok':
    print(Path('/content/oss4008_streamlit.log').read_text()[-4000:])
    raise RuntimeError('Streamlit failed to start; inspect logs above')


## Open the app via Cloudflare

Do **not** use `localhost` or Colab's port-preview link. The previous attempt stalled because of the browser WebSocket connection. The link below will be public while the runtime is running. Close the tunnel when done.


In [ ]:
import os, re, subprocess, time, urllib.request
from pathlib import Path
CLOUDFLARED = Path('/content/cloudflared')
if not CLOUDFLARED.exists():
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        CLOUDFLARED)
    CLOUDFLARED.chmod(0o755)
if 'tunnel_proc' in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate()
    tunnel_proc.wait(timeout=10)
tunnel_log = open('/content/oss4008_cloudflare.log', 'w')
tunnel_proc = subprocess.Popen([str(CLOUDFLARED), 'tunnel', '--url',
    'http://127.0.0.1:8501', '--no-autoupdate'],
    stdout=tunnel_log, stderr=subprocess.STDOUT)
tunnel_url = None
for _ in range(35):
    time.sleep(2)
    logs = Path('/content/oss4008_cloudflare.log').read_text(errors='replace')
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', logs)
    if match:
        tunnel_url = match.group(0)
        break
    if tunnel_proc.poll() is not None:
        break
if tunnel_url:
    print('OPEN YOUR PHASE III APP:', tunnel_url)
    from IPython.display import display, HTML
    display(HTML(f'<a href="{tunnel_url}" target="_blank" rel="noopener noreferrer">Open Residential AI Mapper</a>'))
else:
    print('Tunnel did not start. Log output:')
    print(logs[-5000:])


## Troubleshooting / shutdown

If the link fails, run the diagnostic cell. To close the public link, run the shutdown cell. Colab runtimes are temporary, so rerun the notebook after reconnecting; the ZIP remains in Drive.


In [ ]:
from pathlib import Path
print('STREAMLIT LOG:')
print(Path('/content/oss4008_streamlit.log').read_text(errors='replace')[-4000:])
print('CLOUDFLARE LOG:')
print(Path('/content/oss4008_cloudflare.log').read_text(errors='replace')[-4000:])


In [ ]:
# OPTIONAL: Run only when you are finished testing.
if 'tunnel_proc' in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate()
    print('Public tunnel stopped.')
else:
    print('No active tunnel.')
